In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

base_path = '/content/drive/MyDrive/honours_data'
print(os.listdir(base_path))

['train.csv', 'val.csv', 'test.csv']


In [3]:
pip install transformers datasets torch


In [4]:
import pandas as pd

base_path = '/content/drive/MyDrive/honours_data'

train_df = pd.read_csv(f'{base_path}/train.csv')
val_df   = pd.read_csv(f'{base_path}/val.csv')
test_df  = pd.read_csv(f'{base_path}/test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Train: 185659 | Val: 23207 | Test: 23208


In [5]:
from transformers import DistilBertTokenizer

# load pretrained tokeniser
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# fill nulls before tokenising
train_df['clean_text'] = train_df['clean_text'].fillna('')
val_df['clean_text']   = val_df['clean_text'].fillna('')
test_df['clean_text']  = test_df['clean_text'].fillna('')

# tokenise all three splits
# truncation=True cuts posts longer than 128 tokens
# padding=True pads shorter posts to 128 tokens
def tokenise(texts):
    return tokenizer(
        list(texts),
        max_length=128,
        truncation=True,
        padding=True,
        return_tensors='pt'
    )

print("tokenising train...")
train_enc = tokenise(train_df['clean_text'])
print("tokenising val...")
val_enc   = tokenise(val_df['clean_text'])
print("tokenising test...")
test_enc  = tokenise(test_df['clean_text'])

print("done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenising train...
tokenising val...
tokenising test...
done


In [6]:
import torch
from torch.utils.data import Dataset

class RedditDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # get input_ids and attention_mask for this sample
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

# create datasets for all three splits
train_dataset = RedditDataset(train_enc, list(train_df['label']))
val_dataset   = RedditDataset(val_enc,   list(val_df['label']))
test_dataset  = RedditDataset(test_enc,  list(test_df['label']))

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 185659 | Val: 23207 | Test: 23208


In [7]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# load pretrained distilbert with a classification head (2 labels)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# function to compute metrics during evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# training settings
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/honours_data/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='/content/drive/MyDrive/honours_data/logs',
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("model and trainer ready")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


model and trainer ready


In [8]:
# this will take 30-45 mins on the t4 gpu
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.072977,0.059907,0.979230,0.979230
2,0.028838,0.077552,0.979618,0.979617
3,0.009263,0.087719,0.980135,0.980135


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=17406, training_loss=0.04480798260759214, metrics={'train_runtime': 6102.0937, 'train_samples_per_second': 91.276, 'train_steps_per_second': 2.852, 'total_flos': 1.844532357530573e+16, 'train_loss': 0.04480798260759214, 'epoch': 3.0})

In [9]:
from sklearn.metrics import classification_report

# get predictions on test set
predictions = trainer.predict(test_dataset)
test_preds = np.argmax(predictions.predictions, axis=1)
test_labels = predictions.label_ids

print(f"Test accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=['non-suicide', 'suicide']))

Test accuracy: 0.9809

              precision    recall  f1-score   support

 non-suicide       0.98      0.98      0.98     11604
     suicide       0.98      0.98      0.98     11604

    accuracy                           0.98     23208
   macro avg       0.98      0.98      0.98     23208
weighted avg       0.98      0.98      0.98     23208



In [10]:
save_path = '/content/drive/MyDrive/honours_data/distilbert_model'

# save model and tokeniser
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# save test predictions as csv
results_df = pd.DataFrame({
    'label': test_labels,
    'prediction': test_preds
})
results_df.to_csv('/content/drive/MyDrive/honours_data/distilbert_predictions.csv', index=False)

print(f"model saved to {save_path}")
print("predictions saved to honours_data/distilbert_predictions.csv")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved to /content/drive/MyDrive/honours_data/distilbert_model
predictions saved to honours_data/distilbert_predictions.csv
